# Notebook 081: Shuffle Partition Sizing and Tuning for Small vs Terabyte-Scale Jobs

**Module:** Module 08: Performance Tuning, Query Optimization & Skew Mitigation  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 13: Databricks Clusters & Video 15: Job Run Cost Calculation  

---

## 1. Business Scenario & Real-World Context
A 100MB dataset query takes 45 seconds because Spark defaults to 200 shuffle partitions (`spark.sql.shuffle.partitions = 200`), launching 200 tasks that each process 500KB. Conversely, a 1TB join fails with executor GC timeouts because 200 partitions result in 5GB per task. The team needs a formula to tune shuffle partition sizing.

## 2. Technical Requirement & Problem Statement
Master shuffle partition sizing: calculate optimal partition counts based on shuffle input data size (targeting 100MB - 200MB per partition), configure `spark.sql.shuffle.partitions` dynamically, and compare execution metrics.

## 3. Spark & Databricks Architecture Notes
Spark's default `spark.sql.shuffle.partitions = 200` was chosen arbitrarily in Spark 1.0. If shuffle partitions are too high, task scheduling overhead dominates. If too low, partition data exceeds executor RAM, triggering costly disk spills or OOM errors. Rule of thumb: $ShufflePartitions = ShuffleDataSize / 150MB$.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_081_Shuffle_Partition_Sizing_and_Tuning_for_Small_vs_Terabyte-Scale_Jobs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Sizing Formula Cheatsheet
Review mathematical guidelines for sizing shuffle partitions.

In [ ]:
# Tuning Formula:
# Target Partition Size = 100MB to 200MB
# Shuffle Partitions = Total Input Data at Shuffle / 150MB
# Example: 10GB dataset -> 10,000MB / 150MB ≈ 67 partitions
# Example: 500GB dataset -> 500,000MB / 150MB ≈ 3,333 partitions

print("Shuffle partition calculation guidelines loaded.")

## 6. Benchmarking Small Query with Default (200) vs Tuned (4) Partitions
Observe performance differences on small dataset.

In [ ]:
import time

df_small = spark.range(1, 50000, 1, 4).withColumn("group", F.col("id") % 10)

# Run with default 200 shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "200")
t0 = time.time()
c1 = df_small.groupBy("group").count().collect()
t_default = time.time() - t0
print(f"200 Partitions Execution Time: {t_default:.3f}s (High task scheduling overhead)")

# Run with tuned 4 shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "4")
t0 = time.time()
c2 = df_small.groupBy("group").count().collect()
t_tuned = time.time() - t0
print(f"4 Partitions Execution Time: {t_tuned:.3f}s (Speedup: {t_default/max(t_tuned, 0.001):.1f}x)")

## 7. Dynamic Partitioning via Adaptive Query Execution (AQE)
Enable AQE to automate partition sizing without manual guessing.

In [ ]:
# With AQE enabled, Spark automatically coalesces post-shuffle partitions to ~64MB target size
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", str(64 * 1024 * 1024)) # 64MB advisory size

print("AQE advisory partition size configured to 64MB!")

## 9. Key Architectural Takeaways
- **Target between 100MB and 200MB of compressed data per shuffle partition task.**
- **Default 200 shuffle partitions hurts small queries (< 1GB) and crashes large queries (> 500GB).**
- **Use AQE (`spark.sql.adaptive.advisoryPartitionSizeInBytes`) to let Spark dynamically calculate optimal partition counts.**

## 10. Senior Data Engineering Interview Questions
1. How do you determine the optimal value for `spark.sql.shuffle.partitions` for a 500GB dataset?
2. What happens if `spark.sql.shuffle.partitions` is set too low versus set too high?
3. How does `spark.sql.adaptive.advisoryPartitionSizeInBytes` interact with shuffle partitions?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
